In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score


In [2]:
print("1. Loading and preparing dataset...")
df = pd.read_csv('incidents-wise.csv')

1. Loading and preparing dataset...


### Feature Engineering

In [3]:
df['Incident on'] = pd.to_datetime(df['Incident on'])
df['Year'] = df['Incident on'].dt.year
df['Month'] = df['Incident on'].dt.month
df['Day'] = df['Incident on'].dt.day

In [4]:
df['House_Destroyed_Target'] = (df['House destroyed'] > 0).astype(int)

In [5]:
features = ['Province', 'District', 'Municipality', 'Ward', 'Year', 'Month', 'Day']
X = pd.get_dummies(df[features], drop_first=True)
y = df['House_Destroyed_Target']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
print("\n2. Training and Evaluating Multiple Models...")
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}
model_performances = {}


2. Training and Evaluating Multiple Models...


In [10]:
for name, model in models.items():
    print(f"\n--- Training {name} ---")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    try:
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = 0.0


--- Training Logistic Regression ---

--- Training Random Forest ---

--- Training Gradient Boosting ---


In [12]:
model_performances[name] = {
        "model": model,
        "accuracy": acc,
        "auc": auc
    }
    
print(f"Accuracy: {acc:.4f} | ROC-AUC: {auc:.4f}")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.9696 | ROC-AUC: 0.7536
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       257
           1       0.25      0.17      0.20         6

    accuracy                           0.97       263
   macro avg       0.62      0.58      0.59       263
weighted avg       0.96      0.97      0.97       263



In [13]:
best_model_name = max(model_performances, key=lambda k: model_performances[k]['auc'])
best_model = model_performances[best_model_name]['model']

print(f"\n🏆 Best Performing Model: {best_model_name}")


🏆 Best Performing Model: Gradient Boosting


In [14]:
print("\n3. Making a Prediction using the Best Model...")
sample_data = pd.DataFrame([{
    'Province': 'Bagmati',
    'District': 'Sindhupalchok',
    'Municipality': 'Jugal',
    'Ward': 2,
    'Year': 2026,
    'Month': 9,
    'Day': 15
}])


3. Making a Prediction using the Best Model...


In [15]:
sample_encoded = pd.get_dummies(sample_data)
sample_encoded = sample_encoded.reindex(columns=X.columns, fill_value=0)

prediction = best_model.predict(sample_encoded)
probability = best_model.predict_proba(sample_encoded)

print(f"Sample Incident Prediction: {'House Destroyed' if prediction[0] == 1 else 'No House Destroyed'}")
print(f"Confidence: {probability[0][prediction[0]]:.2f}")

Sample Incident Prediction: No House Destroyed
Confidence: 0.98
